# 05 — DeBERTa v3 + LoRA / optional 4-bit QLoRA

**Important:** DeBERTa requests 4-bit QLoRA, but automatically falls back to ordinary LoRA on macOS/Apple Silicon or unsupported hardware.

In [ ]:
# project setup
from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/voxforge-ai-review-analytics"
).resolve()

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Could not find src/ inside {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%cd /content/voxforge-ai-review-analytics

print("Project root:", PROJECT_ROOT)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

In [ ]:
# verify the runtime
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Current directory:", Path.cwd())
print("src exists:", Path("src").exists())
print("Project files:", list(Path.cwd().iterdir())[:10])

In [ ]:
# connect google drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# copy the dataset from google drive
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/voxforge-ai-review-analytics")

DRIVE_DATASET_PATH = Path(
    "/content/drive/MyDrive/VoxForge/data/processed/cleaned_reviews.csv"
)

PROJECT_DATASET_PATH = (
    PROJECT_ROOT / "data" / "processed" / "cleaned_reviews.csv"
)

PROJECT_DATASET_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not DRIVE_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found in Google Drive: {DRIVE_DATASET_PATH}"
    )

shutil.copy2(
    DRIVE_DATASET_PATH,
    PROJECT_DATASET_PATH,
)

print("Dataset copied successfully.")
print("Source:", DRIVE_DATASET_PATH)
print("Destination:", PROJECT_DATASET_PATH)
print("Exists:", PROJECT_DATASET_PATH.exists())

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import *
from src.config import (
    CLEANED_REVIEWS_PATH, CONFUSION_MATRICES_DIR, MODEL_TRACKING_PATH,
    PLOTS_DIR, PREDICTIONS_DIR, RANDOM_STATE, SENTIMENT_MODELS_DIR,
    create_project_directories,
)
from src.data.load import load_csv
from src.sentiment.transformer import (
    TransformerExperimentConfig, evaluate_trained_transformer,
    plot_training_history, prepare_transformer_sentiment_data,
    runtime_device, save_transformer_artifact, save_transformer_predictions,
    train_transformer,
)
from src.tracker import log_experiment

create_project_directories()
print("Runtime device:", runtime_device())

In [ ]:
# Load the cleaned dataset
reviews = load_csv(CLEANED_REVIEWS_PATH)
model_df = prepare_transformer_sentiment_data(reviews)

In [ ]:
# split the cleaned dataset

X_train_val, X_test, y_train_val, y_test = train_test_split(
    model_df["transformer_text"], model_df["sentiment_label"],
    test_size=0.20, random_state=RANDOM_STATE, stratify=model_df["sentiment_label"],
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=RANDOM_STATE, stratify=y_train_val,
)
print(f"Train: {len(X_train):,} | Validation: {len(X_val):,} | Test held out: {len(X_test):,}")

In [ ]:
# model constants
MODEL_DIR = SENTIMENT_MODELS_DIR / "deberta_v3_sentiment"
CONFUSION_PATH = CONFUSION_MATRICES_DIR / "deberta_v3_sentiment.png"
HISTORY_PATH = PLOTS_DIR / "deberta_v3_sentiment_training_history.png"
PREDICTIONS_PATH = PREDICTIONS_DIR / "deberta_v3_sentiment_validation.csv"

In [ ]:
# Configure and train
config = TransformerExperimentConfig(
    model_id="sentiment_03",
    model_name="DeBERTa v3 + LoRA / optional 4-bit QLoRA",
    pretrained_model="microsoft/deberta-v3-small",
    output_dir=MODEL_DIR,
    max_length=256,
    epochs=3,
    train_batch_size=8,
    eval_batch_size=16,
    learning_rate=1e-4,
    gradient_accumulation_steps=2,
    use_lora=True,
    use_4bit=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    lora_target_modules=("query_proj", "value_proj"),
)
config

In [ ]:
run = train_transformer(
    config, X_train, y_train, X_val, y_val
)

In [ ]:
# evaluate and save useful outputs
validation = evaluate_trained_transformer(
    run, X_val, y_val, confusion_matrix_path=CONFUSION_PATH
)
display(pd.DataFrame([validation["metrics"]]))
display(validation["classification_report"])

In [ ]:
saved_model = save_transformer_artifact(run, MODEL_DIR)
saved_predictions = save_transformer_predictions(
    X_val, y_val, validation, PREDICTIONS_PATH
)
saved_history = plot_training_history(run, HISTORY_PATH)
print("Model:", saved_model)
print("Predictions:", saved_predictions)
print("Training plot:", saved_history)
print("Confusion matrix:", CONFUSION_PATH)

In [ ]:
# Update the shared model tracker
tracking = log_experiment(
    model_id=config.model_id,
    model_name=config.model_name,
    model_family="Transformer PEFT",
    features="Contextual token embeddings",
    preprocessing="transformer_text; tokenizer truncation and dynamic padding",
    algorithm="LoRA; 4-bit QLoRA when available",
    pretrained_model=config.pretrained_model,
    dataset=CLEANED_REVIEWS_PATH.name,
    training_rows=len(X_train),
    validation_rows=len(X_val),
    metrics=validation["metrics"],
    training_time_seconds=run["training_time_seconds"],
    inference_time_ms=validation["inference"]["average_inference_ms"],
    epochs=config.epochs,
    batch_size=config.train_batch_size,
    learning_rate=config.learning_rate,
    max_length=config.max_length,
    artifact_path=saved_model,
    hyperparameters={**config.to_dict(), "using_4bit": run["using_4bit"]},
    output_file=MODEL_TRACKING_PATH,
)
display(tracking)

The test split remains untouched. Use it only once after selecting the final model from validation metrics.